# Notebook 01: Data Cleaning, Ingestion & Tier Matching
This notebook ingests the raw Bechdel test crowd ratings and Cornell Movie-Dialogs corpus, applies title normalization and year tolerance matching, logs dropped films, and visualizes dataset funnels.

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
os.chdir(root)
if str(root / 'src') not in sys.path:
    sys.path.insert(0, str(root / 'src'))
import pandas as pd
from bechdel.data.bechdel_loader import load_bechdel_data
from bechdel.data.cornell_parser import parse_cornell_corpus
from bechdel.data.matching import match_cornell_to_bechdel
from bechdel.viz.plots import plot_class_balance_and_tiers

# 1. Load Tier 1 Bechdel Data
b_df = load_bechdel_data()
print(f'Tier 1 (Bechdel) film count: {len(b_df):,}')
display(b_df.head(3))

Tier 1 (Bechdel) film count: 8,190


,imdbid,rating,title,id,year,decade,pass
0,0392728,0,Roundhay Garden Scene,8040,1888,1880,0
1,0000003,0,Pauvre Pierrot,5433,1892,1890,0
2,0132134,0,"Execution of Mary, Queen of Scots, The",6200,1895,1890,0


In [2]:
# 2. Parse Cornell Movie-Dialogs Corpus
c_data = parse_cornell_corpus()
print(f'Cornell Corpus Titles: {len(c_data["titles"]):,}')
print(f'Cornell Corpus Lines: {len(c_data["lines"]):,}')
print(f'Cornell Corpus Conversations: {len(c_data["conversations"]):,}')

Cornell Corpus Titles: 617
Cornell Corpus Lines: 304,446
Cornell Corpus Conversations: 83,097


In [3]:
# 3. Execute Cross-Dataset Matching (Tier 2)
tier1_df, tier2_df = match_cornell_to_bechdel(
    cornell_titles_df=c_data["titles"],
    bechdel_df=b_df,
    year_tolerance=1,
    data_loss_path='reports/data_loss.md'
)
print(f'Tier 2 Matched Film Count: {len(tier2_df):,} ({(len(tier2_df)/len(c_data["titles"]))*100:.1f}% of Cornell)')
display(tier2_df[['cornell_id', 'title_cornell', 'year_cornell', 'bechdel_rating', 'pass']].head(5))

Tier 2 Matched Film Count: 404 (65.5% of Cornell)


,cornell_id,title_cornell,year_cornell,bechdel_rating,pass
0,m0,10 things i hate about you,1999,3,1
1,m3,2001: a space odyssey,1968,0,0
2,m4,48 hrs.,1982,2,0
3,m5,the fifth element,1997,1,0
4,m6,8mm,1999,1,0


In [4]:
# 4. Visualize Dataset Funnel and Class Balance
plot_class_balance_and_tiers(tier1_df, tier2_df)
print('Saved figure to reports/figures/fig01_class_balance_and_data_loss.png')

Saved figure to reports/figures/fig01_class_balance_and_data_loss.png
